# 多模态机器人智能体：把动作变成 token

> 前面各讲的 Agent 都在数字世界里行动：调用工具、搜索资料、生成代码，动作与反馈都以字符串形式流转。这些循环的边界清晰，环境由我们掌控。
>
> 这一讲把 Agent 放进物理世界，面对一台真实机械臂。核心问题只有一个：模型怎么把看到的画面和听到的指令，变成机械臂的具体动作。这一讲会一步步把它讲清楚，并用代码把每一步跑通。

你可能在视频里见过机械狗和人形机器人的演示。演示者说一句“把红色的杯子放到盘子里”，机器人看一眼桌面，伸出机械臂，张开夹爪，把杯子拿起来放好。整个过程里，机器人同时做三件事：看画面、听懂人话、动起来。

前面每一讲的 Agent 都坐在电脑前。它们能写代码、能搜资料、能规划多步任务，但它们的动作只是调用工具，结果以字符串返回；出错之后，重跑一遍就行。机器人完全不同。它的动作是位移、转角和夹爪的开合，结果由传感器的读数返回；动作一旦出错，可能碰坏东西。这是最大的不同：前面的 Agent 生活在数字世界里，机器人生活在物理世界里。

第 1 讲把 Agent 拆成“感知、决策、行动、反馈”四步循环。前面各讲补的是数字世界里的感知、决策和反馈，这一讲补上的是行动，而且是发生在物理世界里的行动。这种让 Agent 拥有一具真实身体、在物理世界里行动的方案，叫具身智能（embodied intelligence）。

换成真实世界还有一个困难：数据很难收集。web 上现成的图文数据以十亿条计，机器人轨迹数据（Open X-Embodiment）只有约一百万条，差了三个数量级。从头训练的策略无法获得 web 模型已经具备的语义常识，因此这一讲的模型要先借用 web 学到的知识，再学习如何控制机器人。

让机器人动起来的核心，是一类同时处理三样信息的模型：摄像头画面、人的指令、机械臂的动作。这类模型叫视觉-语言-动作模型，缩写 VLA。学完这一讲，我们能说清它为什么这样设计，并亲手从零实现它的核心组件。我们分四步走：先定义具身智能与动作空间，再实现 VLA 的核心组件，接着搭建机器人数据的采集与批量处理流程，最后看物理世界的反馈如何让模型持续改进。我们从“具身智能”和“VLA”这两个基本概念讲起。

## 1. 具身智能与 VLA

### VLA 是什么：把看见、听懂、行动放进同一套参数

这一节回答本讲要打的第一块基础：VLA 是什么。要让机器人完成“把杯子放到盘子里”，需要把三样信息放到一起考虑：画面告诉它杯子在哪里，指令告诉它要做什么，动作告诉机械臂怎么动。这三个环节可以拆成三个独立模块，也可以放进同一个模型。

先看传统做法。传统方案把三件事拆成三个独立模块：视觉模块识别物体位置，规划模块计算轨迹，控制模块驱动电机。模块之间靠人工定义的接口传递数据。任务语义一变，比如从“抓杯子”改成“把盘子推到桌角”，接口就要重新设计。

另一种做法是让同一个模型同时读入画面和指令，直接输出动作，中间不经过任何人为设计的信息传递。这种模型叫视觉-语言-动作模型，缩写 VLA（Vision-Language-Action）。VLA 把感知、理解、控制放进同一套参数统一学习，代价是收敛依赖大量数据。

VLA 有一个之前各讲没遇到的困难：动作怎么表示。前面各讲的 Agent，模型输出字符串（工具名、参数、代码），我们自己写解析器把它变成真实调用。机器人任务里，动作是连续数值（位移、转角、夹爪开度），模型没法直接生成一串浮点数。

VLA 的解决办法是把连续动作离散成一个个动作 token，让动作和文字共用同一个词汇表，训练目标保持为预测下一个 token。RT-2 论文标题里的 Action 一词，指的就是这种“动作也是语言”的设计。

这样一来，具身智能的循环端到端成立：读图像、懂指令、出动作、再观察，每一环都由同一套参数驱动。下一步，我们给出具身智能的正式定义和动作空间的具体表示。

这一节回答两个具体问题：具身智能的循环具体指什么，以及一次机器人动作用什么数值表示。前者说清模型在物理世界里怎么运转，后者为代码提供动作的具体格式。

具身智能（embodied intelligence）把 Agent 的循环完整放进物理世界：感知环境（摄像头图像）、理解指令（自然语言）、输出动作（电机指令）、再观察结果（新的图像帧）。循环的结构和前面各讲相同，变化的只是输入和输出都换成了物理信号，不再是字符串。

要让模型输出动作，先得规定一次动作用几个数表示。一台移动操作臂，一次动作由 7 个连续量描述：末端执行器在 x、y、z 三个方向的位移增量（Δpos_x, Δpos_y, Δpos_z）、绕三个轴的旋转增量（Δrot_x, Δrot_y, Δrot_z）、以及夹爪开度（gripper，取值 0 到 1）。

模型不能直接生成连续小数，所以每个连续量按均匀区间切成 256 个等宽小格，每格用一个 0 到 255 的整数编号。这个从连续值到整数编号的过程叫离散化，每个小格叫一个 bin。动作字符串的开头还有一个终止位，标记动作结束。

先手算一个值，体会离散化的含义。夹爪开度 a = 0.4 落在区间 [0, 1] 上，每格宽 1/256 ≈ 0.0039，它落在第

idx = min(255, ⌊a / (1/256)⌋) = ⌊0.4 × 256⌋ ≈ 102

格。把编号变回连续值时取格子的中心：â = (102 + 0.5) / 256 ≈ 0.4004，与真实值 0.4 的误差在半个格子宽以内。

下面先用一张图看数据规模上的反差。

In [ ]:
# web 数据与机器人数据的规模鸿沟
import numpy as np
import matplotlib.pyplot as plt

labels = ["web pairs\n(billions)", "Open X-Embodiment\n(~1M episodes)"]
counts = [5e9, 1.0e6]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, counts, color=["#4C72B0", "#DD8452"])
ax.set_yscale("log")
ax.set_ylabel("scale (log)")
ax.set_title("web data vs robotics data")
for i, v in enumerate(counts):
    ax.text(i, v, f"{v:.1e}", ha="center", va="bottom")
plt.tight_layout()
plt.show()
print("web 图文约 3 个数量级多于机器人轨迹，这是复用 VLM 的核心动机。")

## 2. 视觉-语言-动作模型

这一节把上一节的动作空间变成能跑的模型。上一节说清了动作怎么表示，这一节要做两件事：把“动作是另一种语言”落到具体的 token 字符串，再从一个最小的 VLA 模型开始，从零实现并训练它。真实系统里，这一节做的正是机器人控制器里最核心的部分：让语言模型学会输出动作。

先看 RT-2 怎么构造动作字符串。把 7 个 bin 序号加上终止位，拼成一个以空格分隔的字符串，例如 `"1 128 91 241 5 101 127 200"`。这个字符串和自然语言指令一起进入训练集，模型按问答格式训练：`Q: what action should the robot take to [task]? A:`，答案是动作字符串。

训练目标仍是 next-token prediction。动作 token 和文字 token 在同一个序列里逐位预测，模型不需要专门的输出通道去生成连续数。

动作 token 从哪来，有一个取舍。RT-2 的 PaLI-X 主干给 0 到 1000 的整数各分配了独立 token，bin 序号直接绑定对应整数 token。PaLM-E 主干不具备这个条件，于是占用 256 个最不常用的旧 token。

占用已有 token 的做法叫 symbol tuning：模型不增加任何参数，只是把旧 token 的含义改写为动作 bin，相当于给旧词表里的几个符号重新贴标签。下面手算一条动作变成 token 字符串的完整过程。

### 手算一遍：一条动作如何变成 token 字符串

这一节把离散化公式套到一条真实动作上，手算出完整的 token 字符串，再反解回连续值看误差。看明白这一步，后面代码里的 toy 轨迹就不难懂。

先看一条动作要描述几个数。机械臂末端在三维空间的位置要用 6 个数完整描述：沿 x、y、z 轴的平移增量，加上绕三轴的旋转增量，再加夹爪开度，一共 7 维。下面代码里 ACTION_RANGE 给每维定了范围，我们沿用这套范围，把一条真实动作逐维量化。

离散化按维度独立进行。第 $d$ 维的区间为 $[lo_d, hi_d]$，动作值 $a_d$ 的 bin 序号为

$$\text{idx}_d = \min\Big(255,\ \Big\lfloor \frac{a_d - lo_d}{hi_d - lo_d} \times 256 \Big\rfloor\Big)$$

这个公式做的事情是：先把动作值在区间里的位置归一化到 0 到 1，再乘 256 看它落在第几格，超过 255 就截断到 255。

取机械臂的一次移动：x 方向平移 −0.20 米，y 方向平移 0.15 米，z 方向不动，三个旋转轴都不转，夹爪开度 0.9。写成 7 维动作是

$$a = [-0.20,\ 0.15,\ 0,\ 0,\ 0,\ 0,\ 0.9]$$

逐维代入公式，得到每维的归一化位置和 bin 序号：

| 维度 | 含义 | 范围 | 归一化 $t$ | $t \times 256$ | bin |
|:---|:---|:---|:---|:---|:---|
| 0 | x 平移（米） | $[-0.25, 0.25]$ | $0.05/0.5 = 0.10$ | 25.6 | 25 |
| 1 | y 平移（米） | $[-0.25, 0.25]$ | $0.40/0.5 = 0.80$ | 204.8 | 204 |
| 2 | z 平移（米） | $[-0.25, 0.25]$ | $0.25/0.5 = 0.50$ | 128.0 | 128 |
| 3 | 绕 x 轴旋转（弧度） | $[-0.60, 0.60]$ | $0.60/1.2 = 0.50$ | 128.0 | 128 |
| 4 | 绕 y 轴旋转（弧度） | $[-0.60, 0.60]$ | $0.60/1.2 = 0.50$ | 128.0 | 128 |
| 5 | 绕 z 轴旋转（弧度） | $[-0.60, 0.60]$ | $0.60/1.2 = 0.50$ | 128.0 | 128 |
| 6 | 夹爪开度 | $[0, 1]$ | $0.90/1.0 = 0.90$ | 230.4 | 230 |

以 y 平移这一维走一步：$t = (0.15 - (-0.25)) / (0.25 - (-0.25)) = 0.40/0.50 = 0.80$，乘 256 得 204.8，向下取整得 204。前面再加上终止位 0，这条动作的 token 字符串就是

```text
0 25 204 128 128 128 128 230
```

反解是逆过程，取每个 bin 的中心值 $\hat{a}_d = lo_d + \frac{\text{idx}_d + 0.5}{256}(hi_d - lo_d)$。以 y 平移这一维为例：$\hat{a}_1 = -0.25 + \frac{204.5}{256} \times 0.5 \approx 0.1494$，与真值 0.15 相差约 0.0006 米。每一维的误差都不超过半个 bin 宽：平移维 bin 宽约 0.00195 米，旋转维约 0.00469 弧度，夹爪维约 0.00391。

量化是有代价的。模型只能表达 256 个离散档位，无法输出任意精细的数值，这是精度受限；换来的是动作和文字在同一个词汇表里统一建模，不需要单独的回归头去输出连续值。bin 越多档位越密，但词表和计算量也越大，RT-2 选 256 是分辨率与规模的折中。

下面代码把这条动作放进 toy 轨迹，验证手算结果，并测量往返误差。往返误差的意思是：动作被离散成 bin 再反解回来，和原来的连续值差多少。

In [ ]:
# RT-2 式动作 token 编解码：7 维动作 × 256 bins
import numpy as np

ACTION_DIM = 7
NUM_BINS = 256
# 每维动作范围：前 3 为平移（米），中 3 为旋转（弧度），最后为夹爪开度（0-1）
ACTION_RANGE = np.array([
    [-0.25, 0.25], [-0.25, 0.25], [-0.25, 0.25],
    [-0.60, 0.60], [-0.60, 0.60], [-0.60, 0.60],
    [0.0, 1.0],
], dtype=np.float64)


def discretize(action):
    """把 7 维连续动作映射到 7 个 bin 序号（每维均匀 256 bin）。"""
    lo = ACTION_RANGE[:, 0]
    hi = ACTION_RANGE[:, 1]
    idx = np.floor((action - lo) / (hi - lo) * NUM_BINS)
    return np.clip(idx, 0, NUM_BINS - 1).astype(int)


def to_action_string(bins, terminate=0):
    """bin 序号数组加终止位，拼成 RT-2 式 token 字符串。"""
    return " ".join([str(terminate)] + [str(int(b)) for b in bins])


def parse_action_string(s):
    """解析 token 字符串，返回 (终止位, bin 序号数组)。"""
    nums = [int(x) for x in s.split()]
    return nums[0], np.array(nums[1:])


def detokenize(bins):
    """bin 序号反解回连续值（取 bin 中心）。"""
    lo = ACTION_RANGE[:, 0]
    hi = ACTION_RANGE[:, 1]
    return lo + (bins + 0.5) / NUM_BINS * (hi - lo)


# toy 动作序列：一段夹取轨迹，逐步靠近目标位姿、夹爪从张开到合拢
rng = np.random.default_rng(42)
toy_actions = np.zeros((5, ACTION_DIM))
toy_actions[:, 0] = np.linspace(-0.2, 0.1, 5)
toy_actions[:, 1] = np.linspace(0.15, -0.05, 5)
toy_actions[:, 6] = np.linspace(0.9, 0.1, 5)

print("每步动作 -> token 字符串（终止位 + 7 个 bin 序号）")
for t in range(5):
    print(f"step {t}: {to_action_string(discretize(toy_actions[t]))}")

# 往返验证：decode 回连续值，误差应小于一个 bin 宽
errors = []
for t in range(5):
    bins = discretize(toy_actions[t])
    recon = detokenize(bins)
    errors.append(np.abs(recon - toy_actions[t]).max())
print("各步最大往返误差：", [f"{e:.4f}" for e in errors])
bin_w = (ACTION_RANGE[:, 1] - ACTION_RANGE[:, 0]) / NUM_BINS
assert all(e < bin_w.max() for e in errors), "往返误差应小于最大 bin 宽"
print("关键观察：往返误差在 bin 宽以内，编解码成立。")

**实验：均匀切分 vs 分位数切分。** 上面验证的是 RT-2 式的均匀切分，把区间按最小值和最大值等分。它有一个弱点：假设动作值散布在整个区间。实际数据里，大多数动作值往往挤在一个小范围，偶尔出现一个偏离很远的极值，这个极值叫离群值。一个离群值就能把区间撑大，每个 bin 变宽，分辨率被浪费。OpenVLA 改用分位数切分，让 bin 边界跟着数据的密集程度走。下面代码用含离群值的合成数据对比两种方式的 bin 宽度。

In [ ]:
# 分位数 binning（OpenVLA） vs min-max（RT-2）：抗离群对比
import numpy as np

rng = np.random.default_rng(7)
normal = rng.normal(0.0, 0.05, size=200)
actions = np.concatenate([normal, [0.9]])


def minmax_bin_width(x, num_bins):
    """RT-2 式：按 [min, max] 均匀切分的 bin 宽。"""
    return (x.max() - x.min()) / num_bins


def quantile_bin_width(x, num_bins, lo=1, hi=99):
    """OpenVLA 式：按 [1%, 99%] 分位数切分的 bin 宽。"""
    qlo, qhi = np.percentile(x, [lo, hi])
    return (qhi - qlo) / num_bins


w_minmax = minmax_bin_width(actions, 256)
w_quant = quantile_bin_width(actions, 256)
print(f"min-max 每个 bin 覆盖宽度：{w_minmax:.4f}")
print(f"分位数每个 bin 覆盖宽度：{w_quant:.4f}")
print(f"有效分辨率提升：{w_minmax / w_quant:.1f} 倍")
assert w_quant < w_minmax, "离群动作下分位数 bin 应更窄"
print("关键观察：一个离群动作就把 min-max 的区间撑大，分位数不受影响。")

动作 token 的训练表示已经讲完，还有一个解码时的问题。真实控制机器人时，模型要在推理阶段逐个预测 token。普通问答任务可以输出任意自然语言 token，机器人任务不行——如果 argmax 落在一个文字 token 上，就得到非法动作。

RT-2 的做法是：机器人任务解码时屏蔽词汇表，只允许在动作 token 上采样，称为输出约束（output constraint）。它不改变训练，只在推理时生效：把非动作位置的 logits 置为 −inf，经过 softmax 后这些位置的概率就是 0，永远不会被选中。

### 手算一遍：屏蔽文字 token 后 argmax 落在哪里

这一节用最小的例子，一步步看清屏蔽做了什么。之后代码用 512 个 token 的玩具词表复现同一个结论。

假设词汇表只有 4 个 token：0、1 是动作 bin，2、3 是文字 token。某一步模型输出的 logits 为

$$\text{logits} = [1.0,\ 2.5,\ 3.0,\ 0.5]$$

不屏蔽时，argmax 取数值最大的位置，落在 2。位置 2 是文字 token，而且它的 logits 最大，直接采样会得到一个非法动作。

屏蔽的做法是把非动作位置替换成 $-\infty$：

$$\text{masked} = [1.0,\ 2.5,\ -\infty,\ -\infty]$$

softmax 里 $e^{-\infty}=0$，这两个位置的输出概率直接归零：

$$p = \frac{[e^{1.0},\ e^{2.5},\ 0,\ 0]}{e^{1.0} + e^{2.5}} \approx [0.18,\ 0.82,\ 0,\ 0]$$

现在 argmax 落在位置 1，是一个动作 bin。

模型权重从头到尾没有被改动，变的只是推理时对 logits 的取舍范围，所以这个方法叫输出约束，而不是再训练。下面代码用 512 个 token 的 toy 词表复现同一件事：不屏蔽时 argmax 落在文字 token，屏蔽后落到动作词表内。

In [ ]:
# 解码时的动作 token 词汇表屏蔽（output constraint）
import numpy as np
import torch

VOCAB_SIZE = 512            # 玩具词表：0-255 为动作 bin，256-511 为文字 token
ACTION_IDS = torch.arange(0, 256)

rng = np.random.default_rng(0)
logits = torch.tensor(rng.normal(0.0, 1.0, size=(VOCAB_SIZE,)))
logits[400] = 4.0           # 人为让一个文字 token 的 logit 最高


def unmasked_dist(logits):
    """不做屏蔽，直接 softmax。"""
    return torch.softmax(logits, dim=-1)


def masked_dist(logits, action_ids):
    """屏蔽后只保留动作 token：其余位置 logits 置 -inf。"""
    m = torch.full_like(logits, -float("inf"))
    m[action_ids] = logits[action_ids]
    return torch.softmax(m, dim=-1)


p_full = unmasked_dist(logits)
p_mask = masked_dist(logits, ACTION_IDS)
top_full = int(p_full.argmax())
top_mask = int(p_mask.argmax())

print(f"不屏蔽：argmax token {top_full}（{'文字' if top_full >= 256 else '动作'}）")
print(f"屏蔽后：argmax token {top_mask}（{'文字' if top_mask >= 256 else '动作'}）")
print(f"屏蔽后非动作 token 的总概率：{p_mask[256:].sum().item():.2e}")
assert p_mask[256:].sum().item() < 1e-6, "屏蔽后非动作 token 概率应为 0"
assert top_mask < 256, "屏蔽后应采样到动作 token"
print("关键观察：屏蔽迫使解码只在动作词表内采样，机器人任务的输出保证合法。")

前面几节解决了动作怎么表示、怎么保证输出合法。这一节把这些部件组装成一个完整模型：用 torch 从零实现一个极小的 VLA。真实 VLA 很大，这里用玩具规模，但结构一一对应，跑通之后真实模型也就理解了。

模型由三个组件拼成。第一个是视觉编码器，把图像特征映射成一个向量。真实模型是 SigLIP 加 DINOv2 融合出来的视觉塔，这里用两层全连接简化。

第二个是指令编码器，把指令映射成一个向量。真实模型是 Llama 语言主干，这里用 token embedding 加平均池化。

第三个是动作解码器，输出每个动作档位的分布。真实模型输出 256 类分布，这里输出 7×256 的 logits，7 对应 7 个动作维度，256 对应每维的档位数。

图像向量和指令向量拼接后进动作解码器，交叉熵只在动作 token 上计算，与 OpenVLA 的训练目标一致。拼接是两种模态合流的关键一步，下一节手算它的数值过程。

### 三块怎么合成一个模型：向量拼接的手算

这一节用一个具体样本，把拼接的每一步数值走一遍，看清两种模态怎么合到一个向量里。之后代码会验证同样的过程。维度取代码里的 d_model=32。

一帧图像是 8×8 灰度图，展平成 64 个数值。视觉编码器是两层全连接：第一层把 64 维压到 32 维，第二层保持 32 维，图像变成向量 $v \in \mathbb{R}^{32}$。

指令 "pick the red cup" 分词后是 4 个 token，每个 token 查词表得到 32 维向量，4 个向量取平均得到 $t \in \mathbb{R}^{32}$。拼接后 $[v;\ t] \in \mathbb{R}^{64}$——两种模态的信息就放在同一个向量里。

动作解码器读这个 64 维向量：先经过一层全连接回到 32 维，再线性映射到 $7 \times 256 = 1792$ 个输出，reshape 成 7 行 256 列。第 $j$ 行是第 $j$ 个动作槽位的 logits，对它做 softmax 得到这一维 256 个 bin 上的概率分布。

训练时交叉熵只在 $7 \times 256$ 个动作 logits 上计算，文字 token 不产生监督，这与 OpenVLA 的训练目标一致。

“在一个模型里融合”意味着所有组件共享同一套参数，图像特征、指令特征、动作头被同一个反向传播更新。没有人为规定“先识别、再规划、后控制”的顺序，模型从数据中自己学习特征如何分配：训练样本里“图像亮度 + 指令编号决定目标 bin”，模型就学到这个映射。

In [ ]:
# mini VLA：视觉编码器(MLP) + 指令编码(Embedding) + 动作解码器
import torch
import torch.nn as nn


class MiniVLA(nn.Module):
    """极小的视觉-语言-动作融合模型。

    输入：image_feat 一张 8x8 灰度图像（展平为 64 维），text_ids 指令 token。
    输出：logits[B, ACTION_DIM, NUM_BINS]，每个动作槽位一个 256 类分布。
    """

    def __init__(self, vis_dim=64, vocab_size=8, d_model=32):
        super().__init__()
        self.vis_enc = nn.Sequential(
            nn.Linear(vis_dim, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
        )
        self.text_emb = nn.Embedding(vocab_size, d_model)
        self.head = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(),
            nn.Linear(d_model, ACTION_DIM * NUM_BINS),
        )

    def forward(self, image_feat, text_ids):
        """返回每个动作槽位的 logits。"""
        x = image_feat.view(image_feat.size(0), -1)  # [B, 64] 展平
        v = self.vis_enc(x)                          # [B, d_model]
        t = self.text_emb(text_ids).mean(dim=1)      # [B, d_model]
        h = torch.cat([v, t], dim=-1)                # [B, 2*d_model]
        logits = self.head(h).view(-1, ACTION_DIM, NUM_BINS)
        return logits


m = MiniVLA()
n_params = sum(p.numel() for p in m.parameters())
print("MiniVLA 参数量：", n_params)

**实验：训练 mini VLA。** 上一节的代码定义了模型，这一节给它一个可以学的任务。我们合成一个 toy 数据集：每张 8×8 图像有一个亮度值，每条指令带一个编号，目标动作 bin 由“亮度等级 + 指令编号”共同决定。模型必须同时用上图像和文字的信息才能预测正确。训练 200 步后，看验证集上 7 个动作槽位全部预测正确的比例。

In [ ]:
# 合成 toy 任务：动作由图像亮度与指令共同决定，训练 mini VLA
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

NUM_TEXT = 4
N_SAMPLES = 256


def make_samples(n):
    """生成 (图像, 指令, 目标动作 bin)。目标由粗亮度等级与指令 id 决定。"""
    imgs = np.random.uniform(0.0, 1.0, size=(n, 8, 8)).astype(np.float32)
    texts = np.random.randint(0, NUM_TEXT, size=(n, 1))
    brightness = np.round(imgs.mean(axis=(1, 2)) * 4).astype(int)   # 0..4
    bins = np.stack([
        np.clip(brightness * 30 + texts[:, 0] * 35 + j * 3, 0, 255)
        for j in range(ACTION_DIM)
    ], axis=1).astype(np.int64)
    return (torch.from_numpy(imgs), torch.from_numpy(texts),
            torch.from_numpy(bins))


train = make_samples(N_SAMPLES)
val = make_samples(64)

model = MiniVLA(vocab_size=NUM_TEXT)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
crit = nn.CrossEntropyLoss()


def evaluate(model, data):
    """动作 token 准确率：7 个槽位全部预测正确才算一步对。"""
    imgs, texts, bins = data
    with torch.no_grad():
        logits = model(imgs, texts)
        pred = logits.argmax(dim=-1)
    return (pred == bins).all(dim=1).float().mean().item()


losses, accs = [], []
for step in range(200):
    opt.zero_grad()
    imgs, texts, bins = train
    logits = model(imgs, texts)
    loss = crit(logits.reshape(-1, NUM_BINS), bins.reshape(-1))
    loss.backward()
    opt.step()
    losses.append(loss.item())
    if step % 25 == 0:
        accs.append(evaluate(model, val))

print(f"最后一步 loss = {losses[-1]:.3f}")
print(f"验证集动作 token 全对率 = {accs[-1]:.2%}")

**实验：观察训练曲线与采样。** 上一节的训练留下了 loss 和验证集准确率两条曲线，这一节把它们画出来看收敛情况，再从验证集里采样几条动作 token。采样的意义在于：推理时模型不是总取概率最大的 token，而是按分布随机选，这一步要确认采样结果仍然落在动作词表内，能直接解码成连续动作。

In [ ]:
# 训练曲线与动作 token 采样
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(losses)
ax[0].set_xlabel("step")
ax[0].set_ylabel("loss")
ax[0].set_title("action-token cross entropy")
ax[1].plot(np.arange(0, 200, 25), accs, marker="o")
ax[1].set_xlabel("step")
ax[1].set_ylabel("val action accuracy")
ax[1].set_title("all-7-bins exact match")
plt.tight_layout()
plt.show()

# 采样三条样本的动作 token，并拼成字符串
model.eval()
imgs, texts, bins = val
with torch.no_grad():
    logits = model(imgs[:3], texts[:3])
    sampled = torch.distributions.Categorical(logits=logits).sample()
for k in range(3):
    pred = sampled[k].tolist()
    true = bins[k].tolist()
    print(f"样本 {k}: 预测 {to_action_string(pred)}")
    print(f"         真值 {to_action_string(true)}")
print("关键观察：采样出的 token 都在 0-255，即动作词表内，可直接解码为连续动作。")

训练 VLA 还有一个普通训练没有的细节。前面的模型直接训练就能学会任务，但真实模型要小心一个坑：只训练机器人数据，模型会把之前学到的知识忘掉。

模型先在一大批 web 图文数据上预训练，学会了“杯子是什么”“盘子是什么”这类常识。如果只用机器人数据继续微调，新数据的梯度一层层改写内部参数，会把语言任务积累下来的表示覆盖掉，模型就忘了常识。这种现象叫灾难性遗忘。

RT-2 的办法是 co-fine-tuning：不只在机器人数据上微调，而是把机器人轨迹和原始 web 视觉-语言数据一起微调，并提高机器人数据在每批里的采样权重。语言数据提醒模型别丢掉常识，机器人数据让模型学会动作，两路一起训。

下面用共享主干的两任务 toy 实验复现这个效应。共享部分是文本编码器，语言任务把它读出类别标签，机器人任务把它读出动作。为了让遗忘真实发生，机器人头只用一层线性映射、中间不留非线性——这样动作任务无法靠头部自行解决，必须改写共享的文本特征，机器人梯度才真正碰到语言能力所在的位置。下面的代码先只预训练语言任务，再对比“只训动作”和“co-fine-tuning”两条路线。

In [ ]:
# co-fine-tuning：共享文本编码器 + 语言头 + 动作头，先预训练语言任务
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)


class TwoTask(nn.Module):
    """共享文本编码器，分别接语言头与动作头。

    共享部分把 8 个指令词映射到 3 维瓶颈；lang_head 与 action_head
    都直接线性读取这段特征。动作头不加非线性层，机器人任务必须
    真正改写共享特征，遗忘才会显现。
    """

    def __init__(self, vocab_size=8, k=3):
        super().__init__()
        self.text_emb = nn.Embedding(vocab_size, k)
        self.vis_proj = nn.Linear(1, k)
        self.lang_head = nn.Linear(k, 4)
        self.action_head = nn.Linear(k * 2, ACTION_DIM * NUM_BINS)

    def forward_lang(self, text_ids):
        """语言任务：文本 -> 类别 logits。"""
        h = self.text_emb(text_ids).mean(dim=1)
        return self.lang_head(h)

    def forward_robot(self, text_ids, brightness):
        """机器人任务：文本 + 亮度 -> 动作 logits。"""
        h = self.text_emb(text_ids).mean(dim=1)
        v = self.vis_proj(brightness)
        feat = torch.cat([h, v], dim=-1)
        return self.action_head(feat).view(-1, ACTION_DIM, NUM_BINS)


VOCAB8 = 8
web_ids = torch.arange(VOCAB8).view(-1, 1)
web_label = torch.tensor([i % 2 for i in range(VOCAB8)])


def make_robot_batch(n=64):
    """机器人样本：随机文本 + 粗亮度等级，目标 bin 由 (亮度, text//2) 决定。"""
    texts = np.random.randint(0, VOCAB8, size=(n, 1))
    brightness = np.random.uniform(0.0, 1.0, size=(n, 1)).astype(np.float32)
    b_level = np.round(brightness[:, 0] * 4).astype(int)     # 0..4
    bins = np.stack([
        np.clip(b_level * 30 + (texts[:, 0] // 2) * 40 + j * 3, 0, 255)
        for j in range(ACTION_DIM)
    ], axis=1).astype(np.int64)
    return torch.from_numpy(texts), torch.from_numpy(brightness), torch.from_numpy(bins)


def lang_accuracy(model):
    """语言任务准确率（8 条指令全对的比例）。"""
    with torch.no_grad():
        logits = model.forward_lang(web_ids)
    return (logits.argmax(dim=1) == web_label).float().mean().item()


def robot_accuracy(model):
    """机器人动作准确率（7 槽全对）。"""
    texts, brightness, bins = make_robot_batch(128)
    with torch.no_grad():
        logits = model.forward_robot(texts, brightness)
    return (logits.argmax(dim=-1) == bins).all(dim=1).float().mean().item()


# 预训练：只训语言任务，直到分类正确
model = TwoTask()
opt = torch.optim.Adam(
    list(model.text_emb.parameters()) + list(model.lang_head.parameters()), lr=1e-2)
crit_lang = nn.CrossEntropyLoss()
for step in range(200):
    opt.zero_grad()
    logits = model.forward_lang(web_ids)
    loss = crit_lang(logits, web_label)
    loss.backward()
    opt.step()
print("预训练后语言准确率：", lang_accuracy(model))

In [ ]:
# 两种微调对比：只训动作 vs co-fine-tuning（动作 + 少量语言）
import matplotlib.pyplot as plt

crit = nn.CrossEntropyLoss()

# (a) 只训机器人数据：语言头冻结，共享特征被动作任务拉走
a = TwoTask()
a.load_state_dict(model.state_dict())
for p in a.lang_head.parameters():
    p.requires_grad = False
opt_a = torch.optim.Adam([p for p in a.parameters() if p.requires_grad], lr=3e-2)

# (b) co-fine-tuning：机器人数据与语言数据一起训，语言头不冻结
b = TwoTask()
b.load_state_dict(model.state_dict())
opt_b = torch.optim.Adam(b.parameters(), lr=3e-2)

lang_a, lang_b = [], []
robot_a, robot_b = [], []
for step in range(400):
    # (a) 只喂机器人数据
    opt_a.zero_grad()
    texts, brightness, bins = make_robot_batch(64)
    loss = crit(a.forward_robot(texts, brightness).reshape(-1, NUM_BINS),
                bins.reshape(-1))
    loss.backward()
    opt_a.step()

    # (b) 机器人 45 条 + 语言全量一起训
    opt_b.zero_grad()
    texts, brightness, bins = make_robot_batch(45)
    loss_r = crit(b.forward_robot(texts, brightness).reshape(-1, NUM_BINS),
                  bins.reshape(-1))
    loss_l = crit(b.forward_lang(web_ids), web_label)
    (loss_r + 0.3 * loss_l).backward()
    opt_b.step()

    if step % 50 == 0:
        lang_a.append(lang_accuracy(a))
        lang_b.append(lang_accuracy(b))
        robot_a.append(robot_accuracy(a))
        robot_b.append(robot_accuracy(b))

steps = list(range(0, 400, 50))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(steps, lang_a, marker="o", label="robot-only")
ax[0].plot(steps, lang_b, marker="s", label="co-fine-tuning")
ax[0].set_xlabel("fine-tune step")
ax[0].set_ylabel("language accuracy")
ax[0].set_title("language retention")
ax[0].legend()
ax[1].plot(steps, robot_a, marker="o", label="robot-only")
ax[1].plot(steps, robot_b, marker="s", label="co-fine-tuning")
ax[1].set_xlabel("fine-tune step")
ax[1].set_ylabel("action accuracy")
ax[1].set_title("action accuracy")
ax[1].legend()
plt.tight_layout()
plt.show()
print(f"只训动作：语言 {lang_a[-1]:.2%}，动作 {robot_a[-1]:.2%}")
print(f"co-fine：语言 {lang_b[-1]:.2%}，动作 {robot_b[-1]:.2%}")
print("关键观察：只训动作时语言能力崩塌，co-fine-tuning 保住语言能力，动作任务照常收敛。")

### 灾难性遗忘发生在哪一步

上面的代码已经跑完两种路线。这一节解释遗忘具体发生在哪一步，以及两股梯度是怎么互相拉扯的。

TwoTask 里语言任务与机器人任务共用同一份 text_emb。语言任务的梯度希望文本特征保留“哪个词属于哪一类”，机器人任务的梯度希望特征变成“能算出 7 个动作 bin”的形状，两个方向的更新常常互相拉扯。

只训练机器人数据时，每步反向传播只看机器人任务的梯度。几十步后，共享特征被改写成对动作有用、对语言分类有害的形状，语言信息被梯度覆盖，这就是灾难性遗忘。我们特意让动作头只有一层线性映射、不留非线性，正是为了让动作任务无法靠头部自行解决——它必须改写共享特征，遗忘才会真正显现。

co-fine-tuning 把两类数据放进同一个 batch。每步更新同时看到语言 loss 与动作 loss：语言 loss 提醒共享特征“类别信息还在用”，动作 loss 把特征推向“能控制动作”。两股梯度互相牵制，收敛到两边都基本满足的位置。RT-2 还提高机器人数据在每批里的采样权重，让动作任务学得更快。

上面代码里 (b) 路线一个 batch 放 45 条机器人样本与全部 8 条语言样本，语言 loss 再乘 0.3 权重，就是这个策略的 toy 版本。跑完看语言准确率的对比最直观：只训动作掉到 50%（8 条指令二分任务的随机水平），co-fine-tuning 保持 100%。

## 3. 机器人数据的采集与批量处理

这一节解决数据从哪来的问题。前面两节说清了模型怎么输出动作，但模型要有数据才能训练，机器人数据不可能像 web 文本那样直接下载。这一节讲机器人数据的采集方式，以及原始记录怎么变成模型能用的训练样本。

机器人数据靠遥操作采集。遥操作的意思是操作员远程操控机器人：戴上 VR 头显，或者直接用手拖着机械臂，完成一次任务。整个过程里，系统同时记录摄像头画面、夹爪状态和动作指令。

这样录下的一条完整记录叫一条轨迹，英文是 episode。一条轨迹的结构是：一段语言指令，加上一串图像帧，再加上一串 7 维动作。

Open X-Embodiment 把 70 多个实验室用这种方式积累的数据汇总成统一格式，筛选后约 100 万条轨迹。下面合成一条 toy 轨迹，演示从原始记录到训练样本的转换。

### 一条轨迹长什么样：逐帧对齐的指令、图像、动作

这一节用一个 toy 轨迹，看清“逐帧对齐”到底指什么，以及一帧数据如何变成一个训练样本。

遥操作的本质是录制一次人类演示，但每一帧都带上控制指令。操作员完成任务时，系统以固定频率采样，每个时间步存下一张图像、夹爪状态和此刻的 7 维动作。整段记录是一条轨迹：一句指令、一串图像帧、一串动作，三者在时间上对齐——第 $t$ 帧图像对应第 $t$ 个动作。

用下面的 toy 轨迹看对齐的含义。轨迹有 16 帧，指令是 "pick the red cup"。第 $t$ 帧图像里圆盘的位置不同，第 $t$ 个动作也随之变化：夹爪开度从第 0 帧的 0.95 逐步合拢到第 15 帧的 0.15，位移增量从初始值逐步过渡到目标值。逐帧配好对，模型才能学到“看到这个画面，就该执行这个动作”。

训练时每一帧都变成一个独立样本（图像、指令、动作 token 字符串），一条 16 帧的轨迹就产出 16 个样本。真实数据里一个任务通常有几十到上百帧，Open X-Embodiment 的约 100 万条轨迹，就来自 70 多个实验室用这种采集方式积累的记录。下面的两条代码先合成这条轨迹，再把它转成训练样本。

In [ ]:
# 合成一条遥操作轨迹：图像帧 + 动作 + 指令
import numpy as np

rng = np.random.default_rng(3)
N_FRAMES = 16


def draw_frame(step):
    """画一帧 8x8 图像：一个圆盘从左上角移向右下角。"""
    img = np.zeros((8, 8))
    r = 1.5
    cx = 1.5 + step / (N_FRAMES - 1) * 5.0
    cy = 1.5 + step / (N_FRAMES - 1) * 5.0
    for y in range(8):
        for x in range(8):
            if (x - cx) ** 2 + (y - cy) ** 2 <= r * r:
                img[y, x] = 1.0
    return img


frames = np.stack([draw_frame(t) for t in range(N_FRAMES)])
actions = np.stack([
    np.linspace(-0.2, 0.1, N_FRAMES),
    np.linspace(0.2, -0.1, N_FRAMES),
    np.linspace(-0.1, 0.05, N_FRAMES),
    np.linspace(-0.3, 0.2, N_FRAMES),
    np.linspace(0.1, -0.05, N_FRAMES),
    np.linspace(-0.2, 0.15, N_FRAMES),
    np.linspace(0.95, 0.15, N_FRAMES),
], axis=1)
instruction = "pick the red cup"

episode = {"instruction": instruction, "frames": frames, "actions": actions}
print("episode 结构：",
      {k: (v.shape if hasattr(v, "shape") else v) for k, v in episode.items()})
print("指令：", instruction)

In [ ]:
# 轨迹 -> 训练样本：每帧 (图像, 指令, 动作 token 字符串)
samples = []
for t in range(N_FRAMES):
    tok = to_action_string(discretize(episode["actions"][t]))
    samples.append({"frame": episode["frames"][t],
                    "instruction": instruction, "action_tokens": tok})

print("前 3 个样本的动作 token 字符串：")
for s in samples[:3]:
    print(" ", s["action_tokens"])
print("样本数 = 帧数 =", len(samples))

单个样本已经就绪，训练时还要把它们组成一批。一批样本里，同一位置的长度必须一致。动作 token 固定是 8 个整数，天然等长；图像是定尺寸数组，直接张量化。只有指令文本长度不固定，需要特殊处理。下一节用一个 toy 批次看具体怎么做，并指出一个容易踩的细节。

### 为什么只有指令需要 padding

这一节回答一个问题：为什么只有指令需要 padding。一批样本里三类输入的长度规律不同。动作 token 固定是 8 个整数（终止位加 7 个 bin 序号），天然等长；图像是定尺寸 8×8 数组，直接堆成张量。变长的只有指令文本："pick the red cup" 是 4 个词，"pick the box" 是 3 个词。

处理方式是每句指令补到批内最长，再配一张 attention mask。下面 5 条指令里最长的一句有 4 个词，不足 4 个词的句子在末尾补 0。attention mask 与文本等长，真实 token 位置记 1，padding 位置记 0，模型只看 mask 为 1 的位置。

这里有一个细节恰好被 toy 数据展示出来。代码里 "pick" 的 id 是 0，padding 也填 0，attention mask 用 `pad_ids != 0` 判断时，会把每句开头的 "pick" 一起当成填充位——上面输出的 mask 第一列全是 0。这不会影响本讲演示的结论（padding 演示本身不参与模型训练），但它说明填充位的 id 必须与任何真实 token 不同，真实模型用专门的 padding token id 避免这种冲突。

In [ ]:
# 指令 token 化与批量处理（padding + attention mask）
import numpy as np

WORD2ID = {"pick": 0, "the": 1, "red": 2, "cup": 3, "blue": 4, "pen": 5,
           "box": 6, "bottle": 7}
instructions = [
    "pick the red cup", "pick the blue pen", "pick the box",
    "pick the red box", "pick the bottle",
]


def tokenize(text):
    """指令按词切分并映射到 id。"""
    return [WORD2ID[w] for w in text.split()]


tok = [tokenize(t) for t in instructions]
max_len = max(len(x) for x in tok)
pad_ids = [x + [0] * (max_len - len(x)) for x in tok]
pad_ids = np.array(pad_ids, dtype=np.int64)
attention_mask = (pad_ids != 0).astype(np.int64)

print("pad 后的 id 矩阵：\n", pad_ids)
print("attention mask：\n", attention_mask)
print("batch 形状：", pad_ids.shape)

# 动作 token 固定 8 个整数，无需 padding；统计一条轨迹产出的动作 token 数
print("一条 16 帧轨迹产出的动作 token 数：", N_FRAMES * (ACTION_DIM + 1))
print("关键观察：动作 token 等长，需要 padding 的只有变长的指令文本。")

## 4. 从物理世界学习的反馈

前两节解决了数据从哪来、怎么处理。但光有数据还不够，模型训练完还要知道它能不能真的干活。这一节讲模型在物理世界里持续改进的循环。

训练不是一次完成的。一个典型的落地循环是：遥操作采集一批轨迹 → 训练模型 → 在真实机械臂上闭环评测 → 把失败轨迹修正后回流数据集 → 再训练。

闭环评测的意思是让模型实时干活：机械臂的摄像头读当前画面，模型根据画面和指令输出下一个动作，机器人实际执行，再看结果。每一步都在真实世界里进行，而不是把数据集重放一遍。

很多机器人先在仿真里训练，成本低。但仿真与真实之间有差距：仿真的物理参数、光照、纹理更理想，模型在仿真里的成功不保证在真实环境成功。这个差距叫 sim-to-real（仿真到真实）差距，闭环评测因此必须回到真机。

闭环评测还会暴露数据质量问题。OpenVLA 清洗 Bridge 数据时发现大量全零动作——夹爪没动却被记录成零位移，这类样本直接污染动作 token 的分布。数据清洗本身是“从物理世界学习”的一部分：记录与标签都不完美，需要反馈来甄别。下面代码用一个简化模型演示回流过程：失败样本修正后回到数据集，数据量翻倍，评测成功率按饱和曲线上升。

In [ ]:
# 简化反馈闭环：失败样本修正后回流，数据量翻倍，评测成功率按饱和曲线上升
import numpy as np
import matplotlib.pyplot as plt


def learning_curve(n, acc_max=0.92, k=600):
    """数据量到成功率的饱和关系：acc = acc_max * (1 - exp(-n/k))。"""
    return acc_max * (1 - np.exp(-n / k))


rounds = np.arange(5)
n_data = 200 * (2 ** rounds)
success = learning_curve(n_data)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(rounds, success, marker="o")
ax.set_xlabel("collection round")
ax.set_ylabel("eval success rate")
ax.set_title("failure samples recycled into training")
for r, n, s in zip(rounds, n_data, success):
    ax.annotate(f"n={n}", (r, s), textcoords="offset points",
                xytext=(0, 8), ha="center")
plt.tight_layout()
plt.show()
print("轨迹累计量：", n_data.tolist())
print("评测成功率：", [round(s, 3) for s in success])

最后一个问题：语言推理怎么接上低层控制。前面模型直接输出动作 token，但很多任务需要先想清楚再动手。RT-2 的一个变体把每条数据扩成 "Instruction: I'm hungry. Plan: pick the snack. Action: 1 128 124 ..." 这样的格式，让模型先生成一句自然语言计划，再输出动作 token。这个做法借鉴了思维链（chain-of-thought）的思路，写作 CoT。语言计划承接 web 预训练带来的推理能力，动作 token 则约束在动作词表内。下面用 llm_client 走一遍这个链式格式，并把动作字符串反解回连续动作。

### 为什么先写计划、再写动作

上一节说模型先生成计划再输出动作。这一节解释这个顺序为什么有效，以及动作部分怎么保证合法。

Plan→Action 把一条训练样本从“指令 + 动作”扩成三段：指令、自然语言计划、动作 token。RT-2 的 CoT 变体用模板 "Instruction: ... Plan: ... Action: ..." 增广机器人数据。

关键是计划承接了 web 预训练的能力。web 语料让模型学会“先用语言推理、再给出结论”的链条，而机器人轨迹本身没有推理过程，只有“该做什么动作”。增广让模型在输出动作前先写一句自然语言推理，把“饿了就该拿零食”这类常识显式说出来。语言计划是 web 知识与动作的桥：模型先调用熟悉的语言推理，再落到不熟悉的动作 token。

动作部分仍受输出约束。推理时解码 Action 段只允许在动作 bin 与终止位上采样，保证结果能反解成合法控制量。语言部分可以自由发挥推理能力，动作部分又保证可执行，两者在同一序列里各司其职。

下面代码把指令喂给 llm_client。环境里配置了真实模型接口时，模型按模板输出 Plan 与 Action；接口不可用时，代码用规则兜底，生成可解析的占位输出。最后把 Action 字符串反解回连续动作，验证闭环。

In [ ]:
# 初始化 LLM 客户端（无 key 时自动进入 真实 API 演示）
import os
import sys

_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

client = get_llm()
print("LLM 客户端：", "脚本化示例（占位输出）" if False else "real API")

In [ ]:
# Plan + Action 链式推理：生成计划，再输出动作 token
import re

# 规则式兜底规划器：真实 API 演示下提供可解析的 Plan/Action 占位
FALLBACK = {
    "hungry": ("pick the snack", "0 140 90 220 30 70 200 60"),
    "thirsty": ("pick the cup", "0 120 70 240 20 80 210 80"),
}


def rule_plan(instruction):
    """按关键词返回 (计划, 动作字符串)；识别不到返回 None。"""
    for key, value in FALLBACK.items():
        if key in instruction:
            return value
    return None


def parse_action_part(text):
    """从回复中提取 'Action: <空格分隔数字>' 部分，提取不到返回 None。"""
    m = re.search(r"Action:\s*([\d\s]+)", text)
    return m.group(1).strip() if m else None


instruction = "I am hungry, pick a snack and hand it to me"
prompt = (
    f"Instruction: {instruction}\n"
    "Plan: <一句自然语言计划>\n"
    "Action: <终止位 + 7 个 bin 序号，空格分隔>"
)

reply = client.chat([{"role": "user", "content": prompt}])
print("模型回复：\n" + reply)

if parse_action_part(reply) is None:
    plan, action_str = rule_plan("hungry")
    source = "真实 API 演示使用规则兜底（占位）"
else:
    action_str = parse_action_part(reply)
    m = re.search(r"Plan:\s*(.+)", reply)
    plan = m.group(1).strip() if m else None
    source = "真实 LLM 输出"

print("\n来源：", source)
print("Plan：", plan)
print("Action：", action_str)
term, bins = parse_action_string(action_str)
recon = detokenize(bins)
print("反解后的连续动作（前 3 维，单位米）：", np.round(recon[:3], 3))
print("夹爪开度：", round(float(recon[6]), 3))
assert len(bins) == ACTION_DIM, "动作 token 数应与动作维度一致"
assert int(term) in (0, 1), "终止位应为 0 或 1"
print("关键观察：语言计划与动作 token 在同一序列里生成，动作 token 反解回连续控制量。")

## 小结

这一节所学的内容：

- VLA 把视觉、语言、动作放进同一个模型：图像与指令经过编码后拼成向量，动作头把向量映射到离散动作 token
- 动作被离散化成 token 才能走语言模型的输出通道：7 维动作各分 256 个 bin，编码成 token 字符串、解码回连续值
- 解码时用词汇表屏蔽把采样限制在动作 bin 内，防止模型输出文字而非动作
- 动作头只在动作槽位计算交叉熵，语言与动作共享前几层编码器
- co-fine-tuning 用同一批数据混合语言与动作两路 loss，防止训练动作时灾难性遗忘语言能力
- 遥操作把人类演示录成逐帧对齐的指令、图像、动作，每帧转成一个训练样本
- Plan→Action 让模型先用语言做高层规划，再落成具体动作，承接 web 预训练的推理能力



## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

三道小题都基于本讲写过的代码，先在草稿上补全，再运行对照。

**作业 1：动作 token 离散化与反解**

补全 `quantile_bins_per_dim` 用 1-99 分位数定每维的 bin 区间，并补全 `quantile_discretize` 与 `quantile_detokenize` 完成量化往返。断言验证：往返误差在分位数 bin 宽以内、分位数 bin 明显窄于 min-max bin。

小提示：先按维度取第 1 与第 99 百分位确定区间端点；对越界值用 clip 收拢。

In [ ]:
# 填空：分位数 bin 区间与量化往返
def quantile_bins_per_dim(actions_2d, num_bins=256, lo=1, hi=99):
    """返回每维的分位数 bin 区间宽度数组。"""
    qlo = np.percentile(actions_2d, lo, axis=0)
    qhi = np.percentile(actions_2d, hi, axis=0)
    return (qhi - qlo) / num_bins


def quantile_discretize(action, bin_edges):
    """action: 7 维数组；bin_edges: (下界数组, 上界数组)。"""
    lo, hi = bin_edges
    idx = np.floor((action - lo) / (hi - lo) * NUM_BINS)
    return np.clip(idx, 0, NUM_BINS - 1).astype(int)


def quantile_detokenize(bins, bin_edges):
    """bin 序号反解回 bin 中心值。"""
    lo, hi = bin_edges
    return lo + (bins + 0.5) / NUM_BINS * (hi - lo)


# 构造含离群动作的 toy 数据
rng = np.random.default_rng(5)
normal = rng.normal(0.0, 0.05, size=(200, ACTION_DIM))
outliers = rng.uniform(0.7, 0.9, size=(2, ACTION_DIM))
data = np.concatenate([normal, outliers], axis=0)

w_minmax = (data.max(axis=0) - data.min(axis=0)) / 256
w_quant = quantile_bins_per_dim(data)
assert np.all(w_quant < w_minmax), "离群动作下分位数 bin 应更窄"
print("min-max 平均 bin 宽：", np.mean(w_minmax))
print("分位数平均 bin 宽：", np.mean(w_quant))

qlo = np.percentile(data, 1, axis=0)
qhi = np.percentile(data, 99, axis=0)
edges = (qlo, qhi)
for k in range(5):
    a = np.clip(data[k], qlo, qhi)
    recon = quantile_detokenize(quantile_discretize(a, edges), edges)
    err = np.abs(recon - a).max()
    assert err < w_quant[k], "往返误差应在分位数 bin 宽以内"
print("作业 1 通过：分位数 binning 抗离群，量化往返误差在 bin 宽以内。")

**作业 2：动作 token 字符串编解码**

补全 `string_to_bins` 与 `string_to_action`，让动作字符串能反解回 bin 序号并重建连续动作。断言验证：字符串以空格分隔且元素个数为 8、字符串级往返误差小于最大 bin 宽。

小提示：`str.split` 切分出终止位与 7 个 bin 序号；反解时沿用 bin 中心公式。

In [ ]:
# 填空：字符串级往返（自包含解析实现）
def string_to_bins(s):
    """解析动作字符串，返回 (终止位, 7 个 bin 序号)。"""
    nums = [int(x) for x in s.split()]
    return nums[0], np.array(nums[1:])


def string_to_action(s):
    """解析动作字符串并反解回连续动作。"""
    _, bins = string_to_bins(s)
    lo = ACTION_RANGE[:, 0]
    hi = ACTION_RANGE[:, 1]
    return lo + (bins + 0.5) / NUM_BINS * (hi - lo)


rng = np.random.default_rng(9)
acts = rng.uniform(ACTION_RANGE[:, 0], ACTION_RANGE[:, 1],
                   size=(8, ACTION_DIM))
max_err = 0.0
for a in acts:
    s = to_action_string(discretize(a))
    assert len(s.split()) == ACTION_DIM + 1, "终止位加 7 个 bin 共 8 个整数"
    recon = string_to_action(s)
    max_err = max(max_err, float(np.abs(recon - a).max()))
assert max_err < (ACTION_RANGE[:, 1] - ACTION_RANGE[:, 0]).max() / NUM_BINS
print("作业 2 通过：动作 token 字符串可反解回连续动作，误差小于最大 bin 宽。")

**作业 3：解码时的动作 token 词汇表屏蔽**

补全 `mask_for_robot_task`：把 logits 中非动作 token 的位置置为 −inf，只保留动作 token。断言验证：屏蔽后非动作位置的 logits 为 −inf、softmax 后非动作 token 概率为 0、argmax 落在动作词表内。

小提示：用 `torch.full_like(logits, -inf)` 建模板，再索引赋值放回动作位置的 logits。

In [ ]:
# 填空：动作 token 词汇表屏蔽
import torch


def mask_for_robot_task(logits, action_ids):
    """屏蔽非动作 token，只保留 action_ids 位置的 logits。"""
    masked = torch.full_like(logits, -float("inf"))
    masked[action_ids] = logits[action_ids]
    return masked


V = 512
ACTION_IDS = torch.arange(256)
rng = np.random.default_rng(2)
logits = torch.tensor(rng.normal(0.0, 1.0, size=(V,)))
logits[500] = 5.0                                   # 文字 token 抢到最高分

masked = mask_for_robot_task(logits, ACTION_IDS)
assert torch.isinf(masked[256:]).all(), "非动作位置应为 -inf"
probs = torch.softmax(masked, dim=-1)
assert probs[256:].sum().item() < 1e-6, "非动作 token 概率应为 0"
assert int(masked.argmax()) < 256, "argmax 应落在动作词表内"
print("作业 3 通过：屏蔽后只在动作词表内采样，机器人任务的输出保证合法。")

## 参考资料

- Brohan et al., [RT-2: Vision-Language-Action Models Transfer Web Knowledge to Robotic Control](https://arxiv.org/abs/2307.15818), 2023 — VLA 的开创性工作：动作即 token、co-fine-tuning、web 知识迁移
- Kim et al., [OpenVLA: An Open-Source Vision-Language-Action Model](https://arxiv.org/abs/2406.09246), 2024 — 第一个开源通用 VLA，分位数 binning、LoRA 与量化推理的系统研究
- Brohan et al., [RT-1: Robotics Transformer for Real-World Control at Scale](https://arxiv.org/abs/2212.06817), 2022 — 35M 参数的离散化动作 transformer，RT-2 的基石与数据来源
- Open X-Embodiment Collaboration, [Open X-Embodiment: Robotic Learning Datasets and RT-X Models](https://arxiv.org/abs/2310.08864), 2023 — 70+ 子数据集与跨本体 RT-X，OpenVLA 的训练数据源
- Driess et al., [PaLM-E: An Embodied Multimodal Language Model](https://arxiv.org/abs/2303.03378), 2023 — 具身多模态语言模型，RT-2-PaLM-E 的 VLM 主干
- Physical Intelligence, [π0: A Vision-Language-Action Flow Model](https://www.physicalintelligence.company), 2024 — 用 flow-matching 连续动作头取代离散 token 的另一条路线
- Karamcheti et al., [Prismatic VLMs](https://arxiv.org/abs/2402.07865), 2024 — OpenVLA 的 VLM 主干，SigLIP+DINOv2 融合视觉编码器
- Hu et al., [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685), 2021 — 低秩适配方法，OpenVLA 参数高效微调的基础
- Chi et al., [Diffusion Policy: Visuomotor Policy Learning via Action Diffusion](https://arxiv.org/abs/2303.04137), 2023 — 连续动作扩散策略，OpenVLA 微调对比的基线
- CS329A 课程大纲 [Lecture 16: Multimodal AI Agents in Robotics](https://cs329a.stanford.edu/) — 本讲在课程地图中的定位